<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L05-constrained-decoding/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T07-L05-constrained-decoding/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T07-L05-constrained-decoding/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T07-L05-constrained-decoding/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T07-L05 · Constrained decoding from scratch: a JSON-schema token mask over a BPE vocabulary

**You will build:** a compiler from a small JSON-schema subset to a byte-level state machine,
the token mask that lets a language model sample only tokens the machine can still finish,
the constrained decoding loop around it, and the measurement of what that mask does to the
model's output distribution.

**Time:** ~45 minutes · **Runs on:** a laptop CPU, no GPU, no download, no API key ·
**Prerequisites:** T03-L01-bpe-from-scratch, T07-L01-agent-loop-from-scratch.

There is no neural network here. The lesson trains a small token n-gram model on JSON text,
tokenised with a byte-level BPE vocabulary of the kind you built in T03-L01, because **the
mask is the subject, not the model**: the mask works the same way whatever produces the
logits. It also means every rate and distance this lesson prints belongs to this toy model
and this one schema: they are not a forecast of how a production model would behave, and
section 11 says what is and is not shown.

By the end you will be able to:

1. Implement a compiler from a small JSON-schema subset to a byte-level state machine, one
   that rejects the keywords it cannot enforce instead of silently ignoring them.
2. Implement the token mask: for a machine state, the BPE tokens whose whole byte string keeps
   the machine alive, with end-of-sequence allowed only in an accepting state.
3. Implement masked sampling and the constrained generation loop over a token n-gram model
   trained in this lesson.
4. Measure the valid-output rate of unconstrained sampling, of sampling with retries and of
   masked decoding, and the tokens each spends per valid output.
5. Measure how far masked decoding shifts the output distribution away from sampling with
   rejection, from the mask masses along each sampled path.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T03-L01`, `T07-L01`

**This lesson assumes you can already:**

- read a regular expression with character classes, alternation and repetition, and say which strings it matches
- explain that a language model gives a probability to every token in its vocabulary and that sampling picks one in proportion to those probabilities
- read a JSON Schema using type, properties, required and enum, and say which documents it accepts

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Efficient Guided Generation for Large Language Models](https://arxiv.org/abs/2307.09702) · Willard and Louf (arXiv) · *free* — section 3, Iterative FSM Processing and Indexing: generation as transitions of a finite-state machine, with a precomputed map from machine states to the vocabulary tokens allowed next
- [Regular-expression derivatives reexamined](https://www.khoury.northeastern.edu/home/turon/re-deriv.pdf) · Owens, Reppy and Turon (Journal of Functional Programming) · *free* — Brzozowski derivatives, the technique section 3's given state machine uses to step a regular expression one byte at a time
- [Models of Computation, lecture 2: Regular Languages](https://jeffe.cs.illinois.edu/teaching/algorithms/models/02-regular.pdf) · Jeff Erickson (University of Illinois) · *free* — 2.1 to 2.3 on languages and regular expressions, and 2.8, Not Every Language is Regular: why a schema without recursion is a regular language and unbounded nesting is not
- [Speech and Language Processing, chapter 3: N-gram Language Models](https://web.stanford.edu/~jurafsky/slp3/3.pdf) · Jurafsky and Martin (Stanford; free draft) · *free* — n-gram models and sampling sentences from a language model, the toy model section 2 trains and masks
- [JSON Schema reference (formerly Understanding JSON Schema)](https://json-schema.org/understanding-json-schema/reference) · JSON Schema project · *free* — the pages on type, object properties and required, and enum: the subset of keywords exercise 1 compiles to a regular expression

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import functools
import io
import json
import os
import random
import signal
import sys
import threading
import time
import traceback
from collections import Counter
from typing import Any, Callable, NamedTuple

print("python", sys.version.split()[0])

SEED = 20260929          # this lesson's one fixed seed: every document, every sample and every
                          # retry derives from it, so re-running prints the same numbers anywhere.

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("schema_to_regex",),
    "exercise 2": ("token_mask",),
    "exercise 3": ("sample_masked",),
    "exercise 4": ("generate",),
    "exercise 5": ("mask_shift",),
}


# What to look at when a check is stopped by the time limit below.
_HANG_HINTS: dict[str, str] = {
    "exercise 1": " A bound is unrolled a fixed number of times, and a bound of 0 is a bound.",
    "exercise 3": " Sampling is one pass over the allowed ids, with the draw you were given.",
    "exercise 4": " generate takes at most max_tokens steps and raises ValueError on an empty mask.",
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (sample_masked)"; several -> "exercises 3, 4 and 5"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# How long an exercise's check may run before it counts as stuck, and how long any other cell that runs your
# code (a demonstration, a table) may: a right answer needs a small fraction of either, measured on the worked
# solution (internal/retrofit_guard.py, 10x or more). tools/verify_all.py reads both numbers.
_CHECK_SECONDS = 2
_DEMO_SECONDS = 45


class _OutOfTime(BaseException):
    """Raised inside a guarded block when its time is up. A BaseException, so an `except
    Exception:` in the code under test cannot swallow it, and it repeats every 0.25 s in case a
    bare `except:` does. The guard turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Turn a call that never returns into a failed check that says so, instead of a notebook
    that hangs. It uses the operating system's alarm where there is one (Linux and macOS, in the
    main thread) and does nothing elsewhere. An alarm that was already running (the grader's own
    time limit) keeps its deadline if it is due first and is put back, with the time it had
    left, afterwards. A correct answer finishes far inside the limit, and nothing that is
    printed depends on it."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s and was stopped: a "
               "loop that never reaches its stopping condition?"
               + _HANG_HINTS.get(what, ""))
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        # an exercise check takes well under a second, a measurement a few seconds
        with _time_limit(_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS, label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## 1. What a token mask is

A language model does not write text; it writes **tokens**, one at a time. At each step it
scores every token in its vocabulary, and a sampler picks one. Constrained decoding puts one
step between the scores and the sampler: a **mask** that removes every token which would make
the output impossible to finish as something valid. Remove a token's score (in a real model
you set its logit to minus infinity), renormalise what is left, sample.

Three things make this harder than it sounds, and this lesson is built around them:

- A token is not a character. One BPE token can be a whole run of bytes such as `"Porto",`,
  which closes a string, writes a comma and moves to the next field, all inside one step. The
  mask has to judge the **whole byte string**, not its first byte.
- "Valid so far" is not enough. A prefix can be perfectly valid and impossible to finish. The
  machine has to know that, and it must allow end-of-sequence **only** when the output is
  complete.
- Renormalising after masking is not the same as asking the model for a valid answer. You
  will measure that difference in section 10.

## 2. The model: a tokenizer and an n-gram, both trained here (given)

The next cell writes a JSON-heavy corpus from a fixed seed: mostly clean calls to a
`get_weather` tool, some sloppy variants (an abbreviated unit, a string where a number
belongs, a missing field) and a few unrelated JSON documents. It then trains a byte-level BPE
on it, using the same algorithm as T03-L01 (a compact copy, so this notebook stands alone),
and a trigram model over the resulting tokens. The slot T03-L01 called `BOUNDARY` is reused
here as the end-of-sequence token, whose bytes are empty.

In [ ]:
BOUNDARY = EOS = 256      # T03-L01's word-boundary slot doubles as end-of-sequence: no bytes
FIRST_MERGE_ID = 257
MIN_COUNT = 2
VOCAB_SIZE = 512
_WHITESPACE = frozenset(b" \t\n\r\f\v")

CITIES = ["Lyon", "Oslo", "Porto", "Kyoto", "Cairo", "Lima", "Turin", "Hanoi", "Perth",
          "Quito", "Seville", "Nairobi", "Rotterdam", "Saint Petersburg", "Buenos Aires",
          "Kuala Lumpur"]
UNITS = ["celsius", "fahrenheit"]
N_DOCS = 400


def make_corpus(n_docs: int, seed: int) -> list[str]:
    """`n_docs` JSON documents, deterministic given `seed`. Given to you."""
    rng = random.Random(seed)
    docs = []
    for _ in range(n_docs):
        r = rng.random()
        args = {"city": rng.choice(CITIES), "unit": rng.choice(UNITS),
                "days": rng.randint(1, 7), "hourly": rng.random() < 0.3}
        if r < 0.70:
            doc: dict = {"name": "get_weather", "arguments": args}
        elif r < 0.80:
            args["unit"] = rng.choice(["C", "F"])                  # sloppy: abbreviated unit
            doc = {"name": "get_weather", "arguments": args}
        elif r < 0.85:
            args["days"] = str(args["days"])                       # sloppy: a string, not a number
            doc = {"name": "get_weather", "arguments": args}
        elif r < 0.90:
            del args["hourly"]                                     # sloppy: a field missing
            doc = {"name": "get_weather", "arguments": args}
        elif r < 0.96:
            doc = {"name": "web_search", "arguments": {
                "query": rng.choice(["rain in Lyon", "best pasta", "train times", "gpu prices"]),
                "limit": rng.randint(1, 10)}}
        else:
            doc = {"id": rng.randint(1, 999), "ok": rng.random() < 0.5, "note": None,
                   "tags": [rng.choice(["a", "b", "c"]) for _ in range(rng.randint(0, 3))]}
        docs.append(json.dumps(doc))
    return docs


def pretokenise(blob: bytes) -> list[int]:
    """Bytes of a word, then BOUNDARY, then the next word. Whitespace itself is dropped."""
    seq, in_word = [], False
    for byte in blob:
        if byte in _WHITESPACE:
            if in_word:
                seq.append(BOUNDARY)
                in_word = False
        else:
            seq.append(byte)
            in_word = True
    if in_word:
        seq.append(BOUNDARY)
    return seq


def merge(seq: list[int], pair: tuple[int, int], new_id: int) -> list[int]:
    """Replace every non-overlapping left-to-right occurrence of `pair` with `new_id`."""
    a, b = pair
    out: list[int] = []
    i, n = 0, len(seq)
    while i < n:
        if i + 1 < n and seq[i] == a and seq[i + 1] == b:
            out.append(new_id)
            i += 2
        else:
            out.append(seq[i])
            i += 1
    return out


def train_bpe(text: str, vocab_size: int) -> tuple[dict, dict]:
    """Byte-level BPE as in T03-L01. Returns (merges, vocab); vocab[id] is that token's bytes."""
    seq = pretokenise(text.encode("utf-8"))
    merges: dict[tuple[int, int], int] = {}
    for k in range(vocab_size - FIRST_MERGE_ID):
        counts: dict[tuple[int, int], int] = {}
        for a, b in zip(seq, seq[1:]):
            if a != BOUNDARY and b != BOUNDARY:
                counts[(a, b)] = counts.get((a, b), 0) + 1
        best, best_n = None, 0
        for pair, n in counts.items():
            if best is None or n > best_n or (n == best_n and pair < best):
                best, best_n = pair, n
        if best is None or best_n < MIN_COUNT:
            break
        merges[best] = FIRST_MERGE_ID + k
        seq = merge(seq, best, FIRST_MERGE_ID + k)
    vocab: dict[int, bytes] = {i: bytes([i]) for i in range(256)}
    vocab[EOS] = b""
    for (a, b), new_id in merges.items():
        vocab[new_id] = vocab[a] + vocab[b]
    return merges, vocab


def encode(text: str, merges: dict) -> list[int]:
    """Text to token ids: merges replayed in training order inside each whitespace-free word."""
    out: list[int] = []
    word: list[int] = []

    def flush() -> None:
        ids = list(word)
        for pair, new_id in merges.items():
            if len(ids) < 2:
                break
            ids = merge(ids, pair, new_id)
        out.extend(ids)

    for byte in text.encode("utf-8"):
        if byte in _WHITESPACE:
            if word:
                flush()
                word = []
            out.append(byte)
        else:
            word.append(byte)
    if word:
        flush()
    return out


def decode(ids: list[int], vocab: dict) -> str:
    """Token ids to text. Bytes that are not valid UTF-8 become the replacement character."""
    return b"".join(vocab[i] for i in ids).decode("utf-8", errors="replace")


class TrigramLM:
    """P(token | previous two tokens), interpolated down to bigram and unigram estimates.

    Every token keeps some probability in every context (that is the smoothing): so this model
    CAN write an invalid document, as an imperfect real model can. Plain floats only, no BLAS.
    """

    def __init__(self, sequences: list[list[int]], vocab_size: int,
                 alpha: float = 1.0, floor: float = 0.1):
        self.vocab_size, self.alpha, self.floor = vocab_size, alpha, floor
        self.uni: Counter = Counter()
        self.bi: Counter = Counter()
        self.tri: Counter = Counter()
        self.bi_total: Counter = Counter()
        self.tri_total: Counter = Counter()
        for seq in sequences:
            padded = [EOS, EOS] + seq
            for a, b, t in zip(padded, padded[1:], padded[2:]):
                self.uni[t] += 1
                self.bi[(b, t)] += 1
                self.bi_total[b] += 1
                self.tri[(a, b, t)] += 1
                self.tri_total[(a, b)] += 1
        self.n_uni = sum(self.uni.values())
        self._cache: dict[tuple[int, int], list[float]] = {}

    def probs(self, context: tuple[int, int]) -> list[float]:
        """The next-token distribution after the two tokens in `context`: V floats summing to 1."""
        if context not in self._cache:
            a, b = context
            V = self.vocab_size
            norm = self.n_uni + self.floor * V
            p1 = [(self.uni[t] + self.floor) / norm for t in range(V)]
            n2 = self.bi_total[b]
            p2 = [(self.bi[(b, t)] + self.alpha * p1[t]) / (n2 + self.alpha) for t in range(V)]
            n3 = self.tri_total[(a, b)]
            self._cache[context] = [(self.tri[(a, b, t)] + self.alpha * p2[t]) / (n3 + self.alpha)
                                    for t in range(V)]
        return self._cache[context]


CORPUS = make_corpus(N_DOCS, SEED)
MERGES, VOCAB = train_bpe("\n".join(CORPUS), VOCAB_SIZE)
SEQUENCES = [encode(doc, MERGES) + [EOS] for doc in CORPUS]
LM = TrigramLM(SEQUENCES, len(VOCAB))

print(f"{len(CORPUS)} documents, {sum(map(len, CORPUS))} bytes -> {sum(map(len, SEQUENCES))} "
      f"tokens ({sum(map(len, SEQUENCES)) / len(SEQUENCES):.1f} per document); vocabulary "
      f"{len(VOCAB)} tokens (256 bytes, end-of-sequence, {len(MERGES)} merges)")
print("one document as tokens:", [VOCAB[t] for t in SEQUENCES[0]])
_after = encode('{"name": "get_weather", "arguments": {"city": "Lyon", "unit":', MERGES) + [32]
_ctx = tuple(([EOS, EOS] + _after)[-2:])
_top = sorted(range(len(VOCAB)), key=lambda t: -LM.probs(_ctx)[t])[:5]
print("after the tokens", [VOCAB[t] for t in _ctx], "the model's likeliest next tokens:",
      [(VOCAB[t].decode(), round(LM.probs(_ctx)[t], 3)) for t in _top])

## 3. The state machine (given): regular expressions with derivatives

A JSON schema with no recursion describes a **regular language**, so a state machine can
recognise it. Rather than build a table by hand, the next cell represents a language as a
small regular expression and uses the Brzozowski derivative: `step(r, b)` is the regular
expression describing what may still follow after one more byte `b`. Its states are
themselves regular expressions, `nullable(r)` says "the output may end here", and the
empty language `EMPTY` is the dead state. The smart constructors keep every language in a
canonical form, so equal states compare equal and the set of reachable states stays finite.

One property carries the whole lesson: **if a state is not `EMPTY`, some continuation still
completes it.** So "does this token keep the machine alive?" and "can this token still be
finished validly?" are the same question.

In [ ]:
EMPTY = ("empty",)
EPS = ("eps",)          # the language containing only the empty string


def chars(byte_values) -> tuple:
    """One byte drawn from `byte_values`. An empty set is the empty language."""
    s = frozenset(byte_values)
    return ("set", s) if s else EMPTY


@functools.lru_cache(maxsize=None)
def _key(r: tuple) -> tuple:
    if r[0] == "set":
        return ("set", tuple(sorted(r[1])))
    return (r[0],) + tuple(_key(x) for x in r[1:])


def cat(a: tuple, b: tuple) -> tuple:
    """`a` then `b`."""
    if a == EMPTY or b == EMPTY:
        return EMPTY
    if a == EPS:
        return b
    if b == EPS:
        return a
    if a[0] == "cat":
        return cat(a[1], cat(a[2], b))
    return ("cat", a, b)


def alt(*rs: tuple) -> tuple:
    """Any one of `rs`."""
    flat: list[tuple] = []
    for r in rs:
        if r[0] == "alt":
            flat.extend(r[1:])
        elif r != EMPTY:
            flat.append(r)
    flat = sorted(set(flat), key=_key)
    if not flat:
        return EMPTY
    return flat[0] if len(flat) == 1 else ("alt",) + tuple(flat)


def star(r: tuple) -> tuple:
    """Zero or more of `r`."""
    if r in (EMPTY, EPS):
        return EPS
    return r if r[0] == "star" else ("star", r)


def lit(text: str) -> tuple:
    """Exactly this text, as UTF-8 bytes."""
    r = EPS
    for byte in reversed(text.encode("utf-8")):
        r = cat(chars([byte]), r)
    return r


def opt(r: tuple) -> tuple:
    """`r` or nothing."""
    return alt(EPS, r)


def upto(r: tuple, n: int) -> tuple:
    """Between zero and `n` copies of `r`."""
    out = EPS
    for _ in range(n):
        out = opt(cat(r, out))
    return out


@functools.lru_cache(maxsize=None)
def nullable(r: tuple) -> bool:
    """May the output end in state `r`?"""
    tag = r[0]
    if tag in ("eps", "star"):
        return True
    if tag in ("empty", "set"):
        return False
    if tag == "cat":
        return nullable(r[1]) and nullable(r[2])
    return any(nullable(x) for x in r[1:])


@functools.lru_cache(maxsize=None)
def step(r: tuple, byte: int) -> tuple:
    """The state after reading one byte: the derivative of `r` by `byte`. EMPTY means dead."""
    tag = r[0]
    if tag in ("empty", "eps"):
        return EMPTY
    if tag == "set":
        return EPS if byte in r[1] else EMPTY
    if tag == "cat":
        rest = cat(step(r[1], byte), r[2])
        return alt(rest, step(r[2], byte)) if nullable(r[1]) else rest
    if tag == "star":
        return cat(step(r[1], byte), r)
    return alt(*[step(x, byte) for x in r[1:]])


def advance(state: tuple, blob: bytes) -> tuple:
    """Feed a run of bytes through the machine, stopping early if it dies."""
    for byte in blob:
        state = step(state, byte)
        if state == EMPTY:
            break
    return state


def accepts(state: tuple, text: str) -> bool:
    """Is `text` a complete member of the language `state` describes?"""
    return nullable(advance(state, text.encode("utf-8")))


def count_states(start: tuple) -> int:
    """How many distinct states the machine can reach, exploring every byte from every state."""
    seen, frontier = {start}, [start]
    while frontier:
        state = frontier.pop()
        for byte in range(256):
            nxt = step(state, byte)
            if nxt != EMPTY and nxt not in seen:
                seen.add(nxt)
                frontier.append(nxt)
    return len(seen)


_demo = cat(lit("ab"), opt(lit("c")))
print("language: ab, then an optional c")
for _text in ["ab", "abc", "a", "abd"]:
    print(f"  {_text!r:6} complete: {accepts(_demo, _text)!s:5}  still alive: "
          f"{advance(_demo, _text.encode()) != EMPTY}")

## 4. Exercise 1 — `schema_to_regex()`

Compile a schema into one of those regular expressions. The subset is deliberately small,
and each rule follows a choice that some production structured-output feature makes (sources
in `claims.yaml`; providers do not all make the same choices, as section 11 shows):

- `{"enum": [...]}`: exactly one of the JSON texts of the listed values.
- `boolean`, `integer` (JSON's own grammar: optional minus, no leading zeros) and `string`
  (printable ASCII except `"` and `\`, so no escapes; optional `maxLength`).
- `array` of `items`, written `[a, b]` (a comma and a space), optional `maxItems` (between
  zero and that many items).
- `object`: every property, in the order the schema lists them, written `{"k": v, "k2": v2}`
  with the same `", "` and `": "` separators as Python's `json.dumps`.
- **Any other keyword raises `ValueError` naming it.** A compiler that quietly ignores
  `minimum` returns a machine that accepts values the schema forbids, while claiming a
  guarantee. (`title`, `description` and `$schema` are annotations: ignore them.)

<details><summary>💡 Hint 1 — what to think about</summary>

Each schema shape becomes one regular expression built from the helpers above, and an object
or an array calls the same function on its parts. What does the text of a two-property object
look like from the first byte to the last? Which pieces are fixed text and which are values?
For integers, what may the first digit be, and what may follow it?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Check for unsupported keywords first, so nothing slips through. Then branch on `enum`, then
on `type`. An object is a chain: the opening brace, then for each property its quoted key, the
colon and space, and the compiled value, with a comma and a space before every property but
the first, then the closing brace. `upto` gives you "at most n copies" for bounded lengths.
</details>

In [ ]:
SUPPORTED_KEYWORDS = {"type", "enum", "properties", "required", "additionalProperties",
                      "items", "maxItems", "maxLength"}
ANNOTATIONS = {"title", "description", "$schema"}
PRINTABLE = [b for b in range(0x20, 0x7F) if b not in (0x22, 0x5C)]   # not '"' and not '\'


def schema_to_regex(schema: dict) -> tuple:
    """Compile a JSON-schema subset into a regular expression over bytes.

    Supported: `enum`; `type` of "boolean", "integer", "string" (optional `maxLength`),
    "array" (with `items`, optional `maxItems`) and "object" (`properties` only: every
    property, in the order listed, separated by ", " and ": ", with no additional keys).
    `required`, if given, must name every property, and `additionalProperties`, if given, must
    be False. Raises ValueError, naming the keyword, for any other keyword.

    Returns the regular expression (a hashable tuple built with the helpers above).

    Example:
        >>> accepts(schema_to_regex({"type": "boolean"}), "true")
        True
        >>> accepts(schema_to_regex({"type": "integer"}), "007")
        False
        >>> schema_to_regex({"type": "integer", "minimum": 0})
        Traceback (most recent call last):
        ValueError: unsupported keyword 'minimum' ...
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_schema_to_regex() -> None:
    day = schema_to_regex({"enum": ["mon", "tue"]})
    assert accepts(day, '"mon"') and not accepts(day, '"wed"'), (
        "an enum accepts exactly the JSON text of its values, quotes included")
    assert not accepts(day, '"mo"'), "a proper prefix of an enum value is not a complete value"
    flag = schema_to_regex({"type": "boolean"})
    assert accepts(flag, "true") and accepts(flag, "false") and not accepts(flag, "True"), (
        "a JSON boolean is the lower-case text true or false")
    num = schema_to_regex({"type": "integer"})
    assert accepts(num, "0") and accepts(num, "-7") and accepts(num, "120"), (
        "an integer may be 0, negative, or several digits")
    assert not accepts(num, "007") and not accepts(num, "") and not accepts(num, "1.5"), (
        "JSON integers have no leading zeros, no empty form and no fraction")
    assert accepts(num, "-0") and not accepts(num, "-01"), (
        "JSON's grammar puts the optional minus in front of either form, so '-0' is an integer, "
        "and the no-leading-zeros rule still applies after the minus ('-01' is not)")
    mixed = schema_to_regex({"enum": [1, "a", True]})
    assert accepts(mixed, "1") and accepts(mixed, '"a"') and accepts(mixed, "true"), (
        "enum values are written as JSON text: a number bare, a string quoted, true lower-case")
    assert not accepts(mixed, '"1"') and not accepts(mixed, "True"), (
        "the quoted text of a number, or Python's spelling of a boolean, is a different value")
    word = schema_to_regex({"type": "string", "maxLength": 3})
    assert accepts(word, '"abc"') and not accepts(word, '"abcd"'), (
        "maxLength is a hard bound: exactly the limit is fine, one more is not")
    assert accepts(word, '""') and not accepts(word, '"ab'), (
        "an empty string is allowed and the closing quote is required")
    nothing = schema_to_regex({"type": "string", "maxLength": 0})
    assert accepts(nothing, '""') and not accepts(nothing, '"a"'), (
        "maxLength 0 is a bound too, allowing only the empty string (a test such as `if limit:` "
        "treats 0 as no bound at all)")
    assert not accepts(word, '"a"b"') and not accepts(word, '"a\\b"'), (
        "an unescaped double quote or backslash cannot appear inside this lesson's strings")
    obj = schema_to_regex({"type": "object", "properties": {
        "z": {"type": "integer"}, "b": {"type": "boolean"}}})
    assert accepts(obj, '{"z": 5, "b": true}'), (
        "an object is {, then each key and value separated by comma-space and colon-space, then }")
    assert not accepts(obj, '{"b": true, "z": 5}') and not accepts(obj, '{"z": 5}'), (
        "properties appear in the order the schema lists them (not sorted), and all of them appear")
    arr = schema_to_regex({"type": "array", "items": {"type": "integer"}, "maxItems": 2})
    assert accepts(arr, "[]") and accepts(arr, "[1, 2]") and not accepts(arr, "[1, 2, 3]"), (
        "an array holds between zero and maxItems items, separated by comma-space")
    for n in (0, 1, 2):
        bounded = schema_to_regex({"type": "array", "items": {"type": "integer"}, "maxItems": n})
        assert accepts(bounded, "[" + ", ".join(["7"] * n) + "]"), (
            f"maxItems={n}: exactly {n} items must be accepted")
        assert not accepts(bounded, "[" + ", ".join(["7"] * (n + 1)) + "]"), (
            f"maxItems={n}: {n + 1} items must be rejected; a bound of 0 allows only [] (a test "
            "such as `if limit:` treats 0 as no bound, and 'the first item, then up to n - 1 "
            "more' still lets one item through when n is 0)")
    for bad, name in [({"type": "integer", "minimum": 0}, "minimum"),
                      ({"enum": [1, 5], "minimum": 3}, "minimum"),
                      ({"$ref": "#"}, "$ref"),
                      ({"type": "object", "properties": {"a": {"type": "integer"}},
                        "minProperties": 1}, "minProperties")]:
        try:
            schema_to_regex(bad)
        except ValueError as exc:
            assert name in str(exc), "the error must name the unsupported keyword"
        else:
            raise AssertionError(f"an unsupported keyword such as {name!r} must raise "
                                 "ValueError, not be ignored (check every schema, including an "
                                 "object that also has properties)")
    props = {"a": {"type": "integer"}, "b": {"type": "integer"}}
    for bad, why in [({"type": "object", "properties": props, "required": ["a"]},
                      "an object whose 'required' omits a property asks for an optional "
                      "property, which this machine cannot express"),
                     ({"type": "object", "properties": props, "additionalProperties": True},
                      "an object that allows additional properties asks for keys the machine "
                      "cannot list")]:
        try:
            schema_to_regex(bad)
        except ValueError:
            pass
        else:
            raise AssertionError(why + ": raise ValueError")


_try("exercise 1", _check_schema_to_regex)

## 5. The schema for this lesson, and its machine

The schema below describes one call to a weather tool. Its `city` has a `maxLength` shorter
than the longest city in the corpus: **on purpose**, so that a bound genuinely binds and section 10
can measure what a bound does to a masked model. (Hosted services do not all enforce such a
bound at all: section 11 has the dated details.)

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {
        "name": {"enum": ["get_weather"]},
        "arguments": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "maxLength": 12},
                "unit": {"enum": ["celsius", "fahrenheit"]},
                "days": {"type": "integer"},
                "hourly": {"type": "boolean"},
            },
        },
    },
}


def start_state() -> tuple:
    return schema_to_regex(SCHEMA)


def _show_machine() -> None:
    start = start_state()
    good = json.dumps({"name": "get_weather", "arguments": {
        "city": "Lyon", "unit": "celsius", "days": 3, "hourly": False}})
    print("a well-formed call is accepted:", accepts(start, good))
    print("a very large value for days is still accepted (nothing bounds an integer):",
          accepts(start, good.replace('"days": 3', '"days": 314159265')))
    print("an unknown unit is rejected:", not accepts(start, good.replace("celsius", "kelvin")))
    print(f"the machine has {count_states(start)} reachable states")


_try("machine demo", _show_machine, needs=("exercise 1",))

## 6. Exercise 2 — `token_mask()`

The heart of constrained decoding. Given the machine's current `state` and the vocabulary
(a dict from token id to that token's bytes, where the end-of-sequence token has empty bytes),
return the ascending list of token ids the sampler may choose from:

- a token with bytes is allowed if feeding **all** its bytes, one after another, never kills
  the machine. Judging only the first byte is the classic bug: a token that starts well and
  ends outside the language is disallowed;
- the empty-bytes token is allowed **only** when `nullable(state)`: the output can end here.

<details><summary>💡 Hint 1 — what to think about</summary>

You already own the tool that feeds bytes through the machine and stops when it dies. What
does it return for a token that leaves the language part-way through? And what should happen
with a token that has no bytes at all: is walking zero bytes through the machine any evidence
it is allowed?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Loop over token ids in ascending order. For a token with bytes, advance the state through all
of them and keep the id unless the result is the dead state. For a token without bytes, keep
the id only if the state is nullable. Return the kept ids as a list.
</details>

In [ ]:
def token_mask(state: tuple, vocab: dict[int, bytes]) -> list[int]:
    """The ids the sampler may choose after `state`, ascending.

    A token with bytes is allowed if walking ALL its bytes from `state` never reaches EMPTY.
    A token with no bytes is end-of-sequence: allowed only if `nullable(state)`.

    Returns a list of token ids, ascending. It is empty if the state is dead.

    Example, language "ab", vocab {0: b"a", 1: b"ab", 2: b"abc", 3: b"b", 4: b""}:
        from the start:      [0, 1]   ("abc" runs past the end, "b" is wrong, 4 is not allowed yet)
        after reading "ab":  [4]      (only end-of-sequence)
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_token_mask() -> None:
    lang = lit("ab")
    toy = {0: b"a", 1: b"ab", 2: b"abc", 3: b"b", 4: b"", 5: b"ba"}
    got = token_mask(lang, toy)
    assert got == [0, 1], (
        f"from the start of the language 'ab' the allowed tokens are 'a' and 'ab' (ids 0 and 1); "
        f"got {got}. Check every byte of a token, in order: 'ba' has two individually plausible "
        f"bytes in the wrong order, and 'abc' runs past the end")
    after_a = advance(lang, b"a")
    assert token_mask(after_a, toy) == [3], (
        f"after 'a' only 'b' can follow (id 3); got {token_mask(after_a, toy)}")
    done = advance(lang, b"ab")
    assert token_mask(done, toy) == [4], (
        f"after 'ab' the output is complete, so only end-of-sequence (id 4) is allowed; got "
        f"{token_mask(done, toy)}. End-of-sequence is allowed in an accepting state and nowhere else")
    assert token_mask(EMPTY, toy) == [], "a dead state allows nothing, not even end-of-sequence"
    shuffled = token_mask(opt(lit("ab")), {9: b"a", 2: b"ab", 5: b"", 7: b"b"})
    assert shuffled == [2, 5, 9], (
        f"return the allowed ids in ascending order, whatever order the vocabulary dict lists "
        f"them in: expected [2, 5, 9], got {shuffled}")


_try("exercise 2", _check_token_mask)

The mask depends only on the state and the vocabulary, and a machine has few states, so it
is cached per state below (given). Here it is on this lesson's schema, at the start and inside
the `city` string.

In [ ]:
class MaskCache:
    """Computes `token_mask` once per machine state."""

    def __init__(self, vocab: dict[int, bytes]):
        self.vocab = vocab
        self._cache: dict[tuple, list[int]] = {}

    def get(self, state: tuple) -> list[int]:
        if state not in self._cache:
            self._cache[state] = token_mask(state, self.vocab)
        return self._cache[state]

    def __len__(self) -> int:
        return len(self._cache)


def _show_mask() -> None:
    start = start_state()
    masks = MaskCache(VOCAB)
    print(f"at the start: {len(masks.get(start))} of {len(VOCAB)} tokens allowed:",
          [VOCAB[t] for t in masks.get(start)][:6])
    prefix = '{"name": "get_weather", "arguments": {"city": "Lyon'
    inside = advance(start, prefix.encode())
    ids = masks.get(inside)
    multi = [VOCAB[t] for t in ids if len(VOCAB[t]) > 1]
    print(f"inside the city string: {len(ids)} allowed, {len(multi)} of them multi-byte, "
          f"for example {multi[:4]}")
    every = [VOCAB[t] for t in ids if b'"' in VOCAB[t][:-1]]    # a quote with bytes after it
    print("tokens that close the string and keep going inside one step:", every[:4])


_try("mask demo", _show_mask, needs=("exercise 1", "exercise 2"))

## 7. Exercise 3 — `sample_masked()`

Given the model's probabilities for every token, the ids the mask allows, and a uniform random
number `u` in [0, 1): return one allowed id, chosen with probability proportional to its
probability **among the allowed tokens only**. That is "set the disallowed logits to minus
infinity and take the softmax", written on probabilities so the lesson prints identical
numbers on every machine. Walk the allowed ids in the order given, add up their probabilities,
and pick the first id whose running total goes past `u` times the total allowed mass. A token
of probability zero must never be picked.

<details><summary>💡 Hint 1 — what to think about</summary>

You need the total probability the mask leaves, and then a way to turn `u` into a position
along that total. Would the answer change if you first divided every allowed probability by
the total? What should happen if the allowed tokens carry no probability at all?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Sum the allowed probabilities; raise an error if the sum is zero. Multiply `u` by that sum to
get a target, then accumulate probabilities in order and return the first token where the
running total is strictly greater than the target, skipping tokens whose probability is zero.
Rounding can leave the loop without a winner when `u` is close to one: return the last
token with positive probability in that case.
</details>

In [ ]:
def sample_masked(probs: list[float], allowed: list[int], u: float) -> int:
    """Sample one id from `allowed`, in proportion to `probs`, using the uniform draw `u`.

    Never returns an id outside `allowed`, or one whose probability is zero. Raises ValueError
    if the allowed ids carry no probability at all.

    Returns one token id.

    Example: probs [0.5, 0.3, 0.2], allowed [1, 2] (id 0 is masked out): the allowed mass is
    0.5, so ids 1 and 2 are chosen with probability 0.6 and 0.4.
        >>> sample_masked([0.5, 0.3, 0.2], [1, 2], 0.1)
        1
        >>> sample_masked([0.5, 0.3, 0.2], [1, 2], 0.9)
        2
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_sample_masked() -> None:
    probs = [0.5, 0.3, 0.2]
    sample_masked(probs, [1, 2], 0.3)
    assert probs == [0.5, 0.3, 0.2], (
        f"sample_masked changed the probs list it was given (now {probs}): leave it alone, since "
        "the model hands out that same list again the next time it sees this context")
    assert sample_masked(probs, [1, 2], 0.05) == 1 and sample_masked(probs, [1, 2], 0.7) == 2, (
        "with id 0 masked, ids 1 and 2 share the remaining mass 0.3:0.2, so u=0.05 lands on 1 "
        "and u=0.7 on 2. Did you sample from the full distribution and ignore the mask?")
    assert sample_masked(probs, [1, 2], 0.5999) == 1 and sample_masked(probs, [1, 2], 0.6001) == 2, (
        "the split between the two allowed ids is at u = 0.3 / 0.5: renormalise over the "
        "allowed ids only, not over all of them")
    picked = {sample_masked([0.0, 0.5, 0.5], [0, 1, 2], u / 100) for u in range(100)}
    assert 0 not in picked, "a token of probability zero must never be sampled"
    assert sample_masked([0.3, 0.3, 0.4], [0, 1, 2], 0.9999999) == 2, (
        "u close to one picks the last allowed id: walk the allowed ids in the order given, not "
        "sorted by probability")
    assert sample_masked([0.5, 0.25, 0.25], [1, 2], 0.5) == 2, (
        "u = 0.5 lands exactly where id 1's share ends (0.25 of the 0.5 left): the running total "
        "must go strictly past the target, so a draw exactly on a boundary belongs to the next id")
    try:
        sample_masked([0.5, 0.5, 0.0], [2], 0.3)
    except ValueError:
        pass
    else:
        raise AssertionError("allowed ids with no probability at all must raise ValueError")


_try("exercise 3", _check_sample_masked)

## 8. Exercise 4 — `generate()`

Now the loop. Each step: take the last two tokens as context (padded on the left with the
end-of-sequence token), ask the model for its distribution, choose the allowed ids, sample
with **one** `rng.random()` draw, stop when the sample is end-of-sequence, otherwise append the
token and advance the machine through its bytes.

- `constrained=True`: allowed ids come from `masks.get(state)`; if that list is empty, raise
  `ValueError` (the machine is stuck). Record `mask_mass`, the **product over steps of the
  probability the model had put on the allowed ids** — section 10 needs it.
- `constrained=False`: every id is allowed, the machine is ignored, `mask_mass` is 1.0.
- Take at most `max_tokens` steps; if end-of-sequence has not come by then, stop with
  `finished=False`. Sampling end-of-sequence is a step too, so a text exactly `max_tokens`
  tokens long whose end-of-sequence was still to come is unfinished.

<details><summary>💡 Hint 1 — what to think about</summary>

Unconstrained sampling is the same as constrained sampling with a mask that allows
everything. What has to differ between the two modes is only which ids are allowed and
whether the machine state and the mass are tracked. Where in the loop can end-of-sequence be
recognised, and should it appear in the returned tokens?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Keep a token list, the machine state, and a running mass starting at 1.0. In each of at most
`max_tokens` iterations: build the context, get the distribution, pick the allowed list,
multiply the mass by the sum of the allowed probabilities (constrained only), sample, and
break with `finished=True` on an empty-bytes token. Otherwise append it and advance the state.
Build the text with `decode` at the end.
</details>

In [ ]:
class Generation(NamedTuple):
    text: str
    tokens: list[int]         # the sampled ids, not including the final end-of-sequence
    finished: bool            # True if end-of-sequence was sampled, False if max_tokens ran out
    mask_mass: float          # product over steps of the probability the mask left (1.0 unconstrained)


def generate(lm, vocab: dict[int, bytes], start: tuple, rng: random.Random, constrained: bool,
             max_tokens: int, masks: MaskCache) -> Generation:
    """Sample one document from `lm`, optionally through the token mask.

    `lm.probs((a, b))` returns the next-token distribution after tokens a, b; the context at
    each step is the last two tokens, padded on the left with EOS. One `rng.random()` per step,
    and at most `max_tokens` steps (sampling end-of-sequence is a step).

    Returns a `Generation`: text, sampled token ids (no end-of-sequence), whether it finished,
    and the mask mass.

    Example (constrained, language "ab", a model that mostly wants to stop at once):
        the output is still "ab": end-of-sequence is not allowed until the machine accepts.
    """
    # YOUR CODE HERE
    raise NotImplementedError


class _CountingRandom(random.Random):
    """A random.Random that counts how often `random()` is called."""

    def __init__(self, seed: int):
        super().__init__(seed)
        self.calls = 0

    def random(self) -> float:
        self.calls += 1
        return super().random()


class _ToyLM:
    """A fixed next-token distribution, whatever the context. Records the contexts it saw."""

    def __init__(self, probs: list[float]):
        self._probs, self.contexts = probs, []

    def probs(self, context: tuple[int, int]) -> list[float]:
        self.contexts.append(context)
        if len(self.contexts) > 2000:
            raise AssertionError("generate asked this toy model for more than 2000 next-token "
                                 "distributions: does your loop stop after max_tokens steps?")
        return list(self._probs)


def _eos_absent(lm, toy: dict, lang: tuple) -> bool:
    """Is the end-of-sequence id absent from a constrained generation's token list?"""
    got = generate(lm, toy, lang, random.Random(11), True, 10, MaskCache(toy))
    return all(toy[t] for t in got.tokens)


def _check_generate() -> None:
    toy = {0: b"a", 1: b"b", 2: b""}                    # id 2 is end-of-sequence
    lang = lit("ab")
    eager = _ToyLM([0.2, 0.2, 0.6])                     # wants to stop 60% of the time
    for seed in range(20):
        got = generate(eager, toy, lang, random.Random(seed), True, 10, MaskCache(toy))
        assert got.text == "ab" and got.finished, (
            f"constrained to the language 'ab', seed {seed} produced {got.text!r} "
            f"(finished={got.finished}). End-of-sequence must not be sampleable until the "
            f"machine accepts, and every sampled token must come from the mask")
    free = generate(_ToyLM([0.0, 0.0, 1.0]), toy, lang, random.Random(0), False, 10, MaskCache(toy))
    assert free.text == "" and free.finished and free.mask_mass == 1.0, (
        "unconstrained, a model that only ever emits end-of-sequence gives the empty text, "
        "finished, with mask_mass 1.0: the machine is ignored")
    runaway = generate(_ToyLM([1.0, 0.0, 0.0]), toy, lang, random.Random(0), False, 5, MaskCache(toy))
    assert runaway.text == "aaaaa" and not runaway.finished, (
        "with no end-of-sequence in sight, generation stops after max_tokens with finished=False")
    seen = _ToyLM([0.2, 0.2, 0.6])
    generate(seen, toy, lang, random.Random(1), True, 10, MaskCache(toy))
    assert seen.contexts[:3] == [(EOS, EOS), (EOS, 0), (0, 1)], (
        f"the context is the last two tokens, padded on the left with EOS; saw {seen.contexts[:3]}")
    counter = _CountingRandom(1)
    steps = generate(_ToyLM([0.2, 0.2, 0.6]), toy, lang, counter, True, 10, MaskCache(toy))
    assert counter.calls == len(steps.tokens) + 1, (
        f"draw exactly one rng.random() per step (one per token, plus one for the final "
        f"end-of-sequence step): expected {len(steps.tokens) + 1} draws, saw {counter.calls}")
    tight = generate(_ToyLM([0.4, 0.4, 0.2]), toy, lang, random.Random(0), True, 2, MaskCache(toy))
    assert tight.text == "ab" and not tight.finished, (
        f"with max_tokens=2 the run takes two steps ('a', 'b') and has no step left for "
        f"end-of-sequence, so it is unfinished; got {tight.text!r}, finished={tight.finished}")
    multi = {0: b"ab", 1: b"c", 2: b"", 3: b"a", 4: b"b"}
    for seed in range(10):
        got = generate(_ToyLM([0.6, 0.1, 0.1, 0.1, 0.1]), multi, lit("abc"), random.Random(seed),
                       True, 10, MaskCache(multi))
        assert got.text == "abc" and got.finished, (
            f"with the two-byte token 'ab' in the vocabulary, seed {seed} produced {got.text!r}: "
            f"after a token, advance the machine through ALL of its bytes, not just the first")
    assert _eos_absent(eager, toy, lang), (
        "the end-of-sequence token is not part of the returned tokens")
    once = generate(_ToyLM([0.5, 0.25, 0.25]), toy, lit("a"), random.Random(3), True, 10, MaskCache(toy))
    assert abs(once.mask_mass - 0.5 * 0.25) < 1e-12, (
        f"mask_mass is the product of the probability left by the mask at every step: "
        f"0.5 for 'a', then 0.25 for end-of-sequence; got {once.mask_mass}")
    try:
        generate(_ToyLM([0.5, 0.5, 0.0]), {0: b"a", 1: b"b", 2: b""}, lit("c"), random.Random(0),
                 True, 5, MaskCache({0: b"a", 1: b"b", 2: b""}))
    except ValueError:
        pass
    else:
        raise AssertionError("a machine that no token can advance must raise ValueError")


_try("exercise 4", _check_generate)

Here is the loop at work, on this lesson's model and schema: unconstrained first, then
constrained, from the same random streams.

In [ ]:
MAX_TOKENS = 80


def is_valid(text: str) -> str:
    """Classify an output: 'valid', 'not JSON', or 'wrong shape' (JSON that breaks the schema)."""
    try:
        value = json.loads(text)
    except ValueError:
        return "not JSON"
    return "valid" if conforms(SCHEMA, value) else "wrong shape"


def conforms(schema: dict, value: Any) -> bool:
    """Does a parsed JSON value satisfy the schema? Full JSON semantics, whatever the spelling.

    (Key order and whitespace do not matter here, so this accepts more than the machine does.)
    """
    if "enum" in schema:
        return any(type(v) is type(value) and v == value for v in schema["enum"])
    kind = schema.get("type")
    if kind == "boolean":
        return type(value) is bool
    if kind == "integer":
        return type(value) is int
    if kind == "string":
        return isinstance(value, str) and len(value) <= schema.get("maxLength", len(value))
    if kind == "array":
        return (isinstance(value, list) and len(value) <= schema.get("maxItems", len(value))
                and all(conforms(schema["items"], v) for v in value))
    if kind == "object":
        props = schema["properties"]
        return (isinstance(value, dict) and set(value) == set(props)
                and all(conforms(props[k], v) for k, v in value.items()))
    return False


def _show_samples() -> None:
    masks, start = MaskCache(VOCAB), start_state()
    for constrained in (False, True):
        print("constrained" if constrained else "unconstrained")
        for i in range(3):
            g = generate(LM, VOCAB, start, random.Random(f"{SEED}:demo:{i}"), constrained,
                         MAX_TOKENS, masks)
            print(f"  {g.text!r}  -> {is_valid(g.text) if g.finished else 'cut off'}")


_try("sample demo", _show_samples, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 9. Measure: valid-output rate, retries and cost

Three ways to get a valid call out of the same model, each over the same number of seeded
requests: sample freely and hope; sample freely and **retry** until the output validates (up
to a limit); mask. The cell counts every token each one generates, because that is what a
real API bills.

In [ ]:
N_SAMPLES = 1500
MAX_RETRIES = 4


def _classify(g: Generation) -> str:
    return is_valid(g.text) if g.finished else "cut off"


def _measure_validity() -> None:
    masks, start = MaskCache(VOCAB), start_state()
    free = [generate(LM, VOCAB, start, random.Random(f"{SEED}:free:{i}"), False, MAX_TOKENS, masks)
            for i in range(N_SAMPLES)]
    kinds = Counter(_classify(g) for g in free)
    print(f"unconstrained, {N_SAMPLES} samples: valid {kinds['valid']} "
          f"({100 * kinds['valid'] / N_SAMPLES:.1f}%), "
          + ", ".join(f"{k} {v}" for k, v in sorted(kinds.items()) if k != "valid"))

    retry_valid = retry_tokens = retry_attempts = 0
    for i in range(N_SAMPLES):
        for attempt in range(MAX_RETRIES + 1):
            g = generate(LM, VOCAB, start, random.Random(f"{SEED}:retry:{i}:{attempt}"), False,
                         MAX_TOKENS, masks)
            retry_tokens += len(g.tokens)
            retry_attempts += 1
            if _classify(g) == "valid":
                retry_valid += 1
                break
    print(f"with up to {MAX_RETRIES} retries: valid {retry_valid} "
          f"({100 * retry_valid / N_SAMPLES:.1f}%), {retry_attempts / N_SAMPLES:.2f} attempts per "
          f"request, {retry_tokens / retry_valid:.1f} tokens generated per valid output")

    con = [generate(LM, VOCAB, start, random.Random(f"{SEED}:con:{i}"), True, MAX_TOKENS, masks)
           for i in range(N_SAMPLES)]
    ckinds = Counter(_classify(g) for g in con)
    print(f"masked: valid {ckinds['valid']} ({100 * ckinds['valid'] / N_SAMPLES:.1f}%), "
          f"{sum(len(g.tokens) for g in con) / ckinds['valid']:.1f} tokens per valid output, "
          f"{len(masks)} distinct masks computed")
    canonical = sum(g.tokens == encode(g.text, MERGES) for g in con)
    print(f"{100 * canonical / N_SAMPLES:.1f}% of masked outputs used exactly the tokens the "
          "tokenizer would have chosen for that text; the rest reached a legal text through a "
          "token sequence the tokenizer never produces for it, so training never showed it")
    steps = binds = allowed_total = 0
    for g in con:
        state = start
        for j, t in enumerate(g.tokens):
            probs = LM.probs(tuple(([EOS, EOS] + g.tokens[:j])[-2:]))
            allowed = masks.get(state)
            steps += 1
            allowed_total += len(allowed)
            binds += max(range(len(probs)), key=lambda k: probs[k]) not in allowed
            state = advance(state, VOCAB[t])
    print(f"on average {allowed_total / steps:.1f} of {len(VOCAB)} tokens were allowed per step; the "
          f"model's own favourite token was masked out at {100 * binds / steps:.1f}% of steps")


_try("validity measurement", _measure_validity,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 10. Exercise 5 — `mask_shift()`: does the mask change what the model would say?

Sampling with rejection (retry until valid, with no retry limit) draws from the model's own
distribution **conditioned on the output being valid**. Masked decoding does not: at each step
it renormalises by `Z`, the probability the model left on allowed tokens, and a path through
steps where `Z` was small is over-sampled compared with the conditioned distribution.

Write `m(x)` for the product of `Z` along one sampled path `x` (that is `mask_mass`).
Then `p(x) / q(x) = m(x)`, where `p` is the model and `q` the masked sampler. Summing over
valid paths gives two facts, both estimated from the masked samples alone:

- the probability that free sampling is valid is `A = E_q[m]`, the mean of `m`;
- the total-variation distance between masked decoding and the conditioned distribution is
  `½ · E_q |1 − m / A|`.

<details><summary>💡 Hint 1 — what to think about</summary>

Everything is an average over the samples you already have. If every sample had the same
`m`, what would `A` be, and what should the distance come out as? What does the distance
look like when the `m` values spread out?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Take the plain mean of the masses for the first number. Then, for each mass, form the ratio
of that mass to the mean, take the absolute difference from one, average those, and halve
the result. Return the pair (A, distance). Raise ValueError for an empty list.
</details>

In [ ]:
def mask_shift(masses: list[float]) -> tuple[float, float]:
    """From the mask masses `m` of masked samples, return (A, tv).

    A  = mean(m): the probability that unconstrained sampling is valid.
    tv = 0.5 * mean(|1 - m / A|): the total-variation distance between masked decoding and
    the model conditioned on being valid (over token paths).

    Returns the pair (A, tv). Raises ValueError for an empty list.

    Example: masses [0.5, 1.5] -> A = 1.0, tv = 0.5 * mean(0.5, 0.5) = 0.25.
        [0.4, 0.4, 0.4] -> (0.4, 0.0): a constant mass means no distortion at all.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_mask_shift() -> None:
    a, tv = mask_shift([0.4, 0.4, 0.4])
    assert abs(a - 0.4) < 1e-12 and abs(tv) < 1e-12, (
        f"equal masses mean masked decoding equals the conditioned distribution: expected "
        f"(0.4, 0.0), got ({a}, {tv})")
    a, tv = mask_shift([0.5, 1.5])
    assert abs(a - 1.0) < 1e-12 and abs(tv - 0.25) < 1e-12, (
        f"for masses [0.5, 1.5] expected A=1.0 and tv=0.25, got ({a}, {tv}). Average over all n "
        f"masses (not n - 1), and divide each mass by "
        f"the MEAN mass before taking |1 - ratio|, then average, then halve")
    a, tv = mask_shift([0.1, 0.1, 0.1, 0.7])
    assert abs(a - 0.25) < 1e-12 and abs(tv - 0.45) < 1e-12, (
        f"for masses [0.1, 0.1, 0.1, 0.7] expected A=0.25 and tv=0.45, got ({a}, {tv})")
    try:
        mask_shift([])
    except ValueError:
        pass
    else:
        raise AssertionError("an empty list has no mean: raise ValueError")


_try("exercise 5", _check_mask_shift)

Now apply it to the masked samples, and check the result against the world: `A` was
computed from the mask alone, so it should match the rate at which free sampling lands
inside the machine's language, up to sampling error: both are estimates from a finite number
of samples, so the cell prints their standard errors too. It then looks at the cities the two
methods produce, and separates the odd cities the schema's bound forces from the ones the
model's own slips produce.

In [ ]:
def _measure_shift() -> None:
    masks, start = MaskCache(VOCAB), start_state()
    con = [generate(LM, VOCAB, start, random.Random(f"{SEED}:con:{i}"), True, MAX_TOKENS, masks)
           for i in range(N_SAMPLES)]
    done = [g for g in con if g.finished]
    masses = [g.mask_mass for g in done]
    a, tv = mask_shift(masses)
    free = [generate(LM, VOCAB, start, random.Random(f"{SEED}:free:{i}"), False, MAX_TOKENS, masks)
            for i in range(N_SAMPLES)]
    inside = sum(g.finished and accepts(start, g.text) for g in free) / N_SAMPLES
    print(f"{len(done)} of {len(con)} masked samples finished within the token limit")
    print(f"A estimated from the masks alone:  {a:.4f}")
    print(f"free samples inside the language:  {inside:.4f}  (counted directly)")
    se_a = (sum((m - a) ** 2 for m in masses) / max(len(masses) - 1, 1)) ** 0.5 / len(masses) ** 0.5
    se_in = (inside * (1 - inside) / N_SAMPLES) ** 0.5
    z = abs(a - inside) / (se_a ** 2 + se_in ** 2) ** 0.5
    print(f"standard errors {se_a:.4f} and {se_in:.4f}: the two estimates differ by {z:.1f} "
          "standard errors of their difference, "
          + ("consistent with sampling noise" if z < 3 else "more than sampling noise explains"))
    print(f"total-variation shift of masked decoding away from rejection sampling: {tv:.3f}, "
          f"that is {100 * tv:.1f}% of the probability mass sitting on different paths")
    print("  (over token paths; two paths can spell the same text, so over texts it can only be "
          "smaller)")
    fifth = len(masses) // 5
    if fifth:
        parts = [mask_shift(masses[k * fifth:(k + 1) * fifth])[1] for k in range(5)]
        print(f"  the same distance on five disjoint fifths of the masked samples: {min(parts):.3f} "
              f"to {max(parts):.3f}, a Monte Carlo estimate, not an exact value")

    retry = []
    for i in range(N_SAMPLES):
        for attempt in range(MAX_RETRIES + 1):
            g = generate(LM, VOCAB, start, random.Random(f"{SEED}:retry:{i}:{attempt}"), False,
                         MAX_TOKENS, masks)
            if g.finished and is_valid(g.text) == "valid":
                retry.append(g)
                break

    def odd_cities(gens: list[Generation]) -> Counter:
        found: Counter = Counter()
        for g in gens:
            city = json.loads(g.text)["arguments"]["city"]
            if city not in CITIES:
                found[city] += 1
        return found

    odd_masked, odd_retry = odd_cities(done), odd_cities(retry)
    print(f"cities that are not in the training corpus: masked {sum(odd_masked.values())} of "
          f"{len(done)} ({100 * sum(odd_masked.values()) / len(done):.1f}%), "
          f"retry {sum(odd_retry.values())} of {len(retry)} "
          f"({100 * sum(odd_retry.values()) / len(retry):.1f}%)")
    print("most common such cities under the mask:", odd_masked.most_common(3))
    bound = SCHEMA["properties"]["arguments"]["properties"]["city"]["maxLength"]
    too_long = [c for c in CITIES if len(c) > bound]
    heads = tuple(c.split()[0] + " " for c in too_long)
    forced = sum(n for c, n in odd_masked.items() if c.startswith(heads))
    print(f"{forced} of the {sum(odd_masked.values())} begin like a corpus city longer than the "
          f"bound ({', '.join(too_long)}): the mask cut the model off mid-name; the rest are the "
          "model's own slips, which the mask made legal and a retry would have thrown away")
    longest = max(len(c) for c in CITIES)
    print(f"the longest real city has {longest} characters and the schema allows "
          f"{SCHEMA['properties']['arguments']['properties']['city']['maxLength']}: the mask "
          "guarantees a legal string, not a sensible one")


_try("shift measurement", _measure_shift,
     needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4", "exercise 5"))

## 11. Where providers differ: what a regular machine cannot say

Your compiler refuses `minimum`, `pattern` and `$ref` rather than pretending. Hosted
structured-output features draw that line in different places, and they change: **as of
2026-09-29** Anthropic's documentation lists recursive schemas, numerical constraints
(`minimum`, `maximum`) and string length constraints (`minLength`, `maxLength`) as not
supported, and answers an unsupported feature with an error, while OpenAI's documentation says
recursive schemas are supported and lists `minimum`, `maximum` and `maxItems` among supported
properties, with a different list for fine-tuned models (`claims.yaml` has each quote).

Two hosted features exist under that heading, and their names are dated facts too. **As of
2026-09-29** Anthropic documents JSON outputs (the `output_config.format` parameter) and
strict tool use (`strict: true` on a tool), and says its schema-compliant responses come from
constrained decoding; OpenAI documents Structured Outputs, switched on with `strict: true`,
and returns an error when the schema uses an unsupported feature.

**What this lesson is not.** It is a teaching model, not a copy of those services. It uses a
small n-gram model rather than a neural network, a tokenizer trained on a few hundred
documents rather than a production vocabulary, and no XGrammar or other production engine, so
it shows the mechanism (a per-step mask, an end-of-sequence rule, a per-step renormalisation)
and not the behaviour of any provider's model. How often a real model breaks a schema without
the mask, and how far a real engine's mask moves its output, depend on that model and on how
the engine works, and nothing here measures them. The documentation quotes in `claims.yaml`
are the only statements about those services.

Why recursion is the awkward one: nesting depth is unbounded, and a finite machine cannot
count. The cell below unrolls a nested list to a fixed depth and counts the machine's states.

In [ ]:
def nested_list_schema(depth: int) -> dict:
    schema: dict = {"type": "integer"}
    for _ in range(depth):
        schema = {"type": "array", "items": schema, "maxItems": 2}
    return schema


def _show_recursion() -> None:
    for depth in range(1, 5):
        print(f"nesting depth {depth}: {count_states(schema_to_regex(nested_list_schema(depth)))} "
              "states")
    for bad in [{"type": "integer", "minimum": 0}, {"$ref": "#"}, {"type": "string", "pattern": "^a"}]:
        try:
            schema_to_regex(bad)
        except ValueError as exc:
            print("refused:", exc)


_try("recursion demo", _show_recursion, needs=("exercise 1",))

## 12. Common mistakes

- **Judging a token by its first byte.** One BPE token spans many bytes; the mask must walk
  all of them, or the output will leave the language mid-token.
- **Allowing end-of-sequence everywhere.** An empty-bytes token passes a "does not kill the
  machine" test trivially, so it needs its own rule: accepting states only.
- **Ignoring a keyword the machine cannot enforce.** The result is a decoder that promises a
  guarantee it does not give.
- **Confusing "valid prefix" with "valid output".** A length limit can cut generation off
  inside a perfectly legal prefix: keep `finished` and count the cut-offs.
- **Assuming the mask leaves the distribution alone.** Renormalising per step is not
  conditioning the whole output on validity.

In [ ]:
print(f"'ba' walked byte by byte through the language 'ab': {advance(lit('ab'), b'b') == EMPTY} "
      "(dead at the first byte)")
print(f"an empty-bytes token walked through any state leaves it unchanged: "
      f"{advance(lit('ab'), b'') == lit('ab')}, which is why it needs an explicit rule")

## 13. Self-check

1. Why must `token_mask` walk every byte of a multi-byte token instead of checking its first?
   - (a) Because only a token's first byte is stored, so the decoder cannot read the rest
   - (b) Because one token can begin validly and end outside the language, and the sampler
     picks whole tokens, so a mask that checks only the start lets invalid output through
   - (c) Because a BPE token is always valid JSON on its own, so one byte of it settles it
   - (d) Because the machine takes one byte per step, so a token must be split across steps

2. A state is not the dead state. Which statement about it is true?
   - (a) The output may end here, because the machine has not died and so must be accepting
   - (b) Every token in the vocabulary is allowed here, because no byte has killed the machine
   - (c) Some sequence of bytes still completes a valid output, though the output need not be
     complete yet
   - (d) The model considers it likely, because the state was reached by sampling from the model

3. Masked decoding and "sample, reject invalid, retry" both only ever return valid outputs.
   Why can their output distributions still differ?
   - (a) Masked decoding renormalises at each step by how much probability survived that step,
     which weights whole paths differently from conditioning the entire output on being valid
   - (b) Retries use a different random number generator, so they draw from another distribution
   - (c) Masked decoding changes the model's weights a little at every step it is applied, so
     later steps see a different model from the one that was trained
   - (d) Rejection sampling only accepts short outputs, so its samples are biased towards them

4. Why does a finite state machine handle a schema with `maxItems` but not a recursive schema?
   - (a) Recursive schemas contain floating-point numbers, which no finite machine can track
   - (b) `maxItems` is a JSON keyword and recursion is not, so only keywords can be compiled
   - (c) Bounded nesting can be unrolled into finitely many states; unbounded nesting would
     need a stack, because the machine would have to count open brackets
   - (d) Recursive schemas are never valid JSON, so no decoder is asked to produce them

5. Masked generation stops at the token limit before end-of-sequence. What do you know?
   - (a) The output is invalid JSON but a valid prefix of the language, so it can be resumed
     or retried but not parsed as it stands
   - (b) The mask failed, because a mask that works never lets a sample run out of tokens
   - (c) The model has a bug, because a working model always ends its output within the limit
   - (d) The output is complete but was truncated by the tokenizer after its last token

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## What you built

A compiler from a schema to a byte-level state machine, the mask that turns that machine into
a per-step list of allowed tokens, and the loop that samples through it. You also measured what
a guarantee costs and does not cost: in this toy setup masked decoding never finishes an
invalid output (it can still be cut off by the token limit), spends no tokens on retries, and
leaves the model's own distribution behind in a way you can estimate from the mask masses
alone. The next lesson on this path moves on from the shape of one call to
the reliability of many.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now: each one already printed its feedback in its own cell above, so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_schema_to_regex, ()),
        ("exercise 2", _check_token_mask, ()),
        ("exercise 3", _check_sample_masked, ()),
        ("exercise 4", _check_generate, ()),
        ("exercise 5", _check_mask_shift, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.